# 07 — Final EGFR Resistance Model Comparison
Compare baseline, enhanced, rich-no-aux, rich-all using grouped CV.

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

In [ ]:
import pandas as pd, numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold,cross_val_predict
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from scipy.stats import spearmanr

data=pd.read_csv('egfr_structural_features_rich.csv')
print(data.shape)
display(data.head())

## Define feature sets

In [ ]:
target='auc_ratio_vs_wt'
groups=data['mutation']

exclude={'mutation','drug','auc_ratio_vs_wt','resistant','source_file','sheet','sample'}
numeric=[c for c in data.columns if c not in exclude and data[c].dtype!='O']

baseline=[c for c in numeric if ('local' not in c and 'auxiliary' not in c and 'drug_' not in c and 'putative' not in c and 'protein_drug' not in c and 'pocket_' not in c and 'steric' not in c and 'sidechain_drug' not in c)]
# use original obvious cols
baseline=[c for c in ['delta_sidechain_volume_A3','delta_hydropathy','delta_charge','binding_site_rmsd_A','contacts_lost','contacts_gained'] if c in data.columns]

enhanced=[c for c in numeric if c not in baseline and ('auxiliary' not in c and not c.startswith('drug_') and 'putative' not in c and 'protein_drug' not in c and 'pocket_' not in c and 'steric' not in c and 'sidechain_drug' not in c)]
enhanced=sorted(set(baseline+enhanced))

rich_no_aux=[c for c in numeric if 'auxiliary' not in c]
rich_all=numeric

feature_sets={
'Baseline':baseline,
'Enhanced':enhanced,
'Rich_no_aux':rich_no_aux,
'Rich_all':rich_all
}
for k,v in feature_sets.items():
    print(k,len(v))


In [ ]:
results=[]
cv=GroupKFold(n_splits=5)
for name,features in feature_sets.items():
    X=data[features+['drug']]
    y=data[target]
    pre=ColumnTransformer([
        ('num',Pipeline([('imp',SimpleImputer(strategy='median'))]),features),
        ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),['drug'])
    ])
    model=Pipeline([('pre',pre),('rf',RandomForestRegressor(n_estimators=300,random_state=1))])
    pred=cross_val_predict(model,X,y,cv=cv,groups=groups,n_jobs=-1)
    results.append({
        'Model':name,
        'MAE':mean_absolute_error(y,pred),
        'RMSE':mean_squared_error(y,pred)**0.5,
        'R2':r2_score(y,pred),
        'Spearman':spearmanr(y,pred).statistic
    })
res=pd.DataFrame(results).sort_values('R2',ascending=False)
display(res)


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.bar(res['Model'],res['R2'])
plt.ylabel('Cross-validated R²')
plt.xticks(rotation=20)
plt.show()